In [6]:
### rag with LLM 
from langchain_community.document_loaders import TextLoader

text = TextLoader("cricket_hist.txt")

data = text.load()

In [7]:
data

[Document(metadata={'source': 'cricket_hist.txt'}, page_content="Cricket's origins can be traced back to the late 16th century in southeast England, with the earliest definite reference to the sport being in 1597. It evolved from earlier bat-and-ball games, potentially involving shepherds using their staffs to deflect a ball. The sport gained popularity in the 18th century, spreading throughout England and then to other parts of the British Empire. The first Laws of Cricket were codified in 1744, and the Marylebone Cricket Club (MCC) became the custodian of the Laws in 1787. International cricket emerged in the mid-19th century, with the first international match played in 1844 between the USA and Canada. \nHere's a more detailed look at the history:\nEarly Development (16th-18th Centuries):\nSaxon/Norman Roots:\nExperts believe cricket originated in the Weald region of southeast England during the Saxon or Norman periods. \nFirst References:\nThe earliest definite reference to the spo

In [8]:
### load the env
from dotenv import load_dotenv
import os
load_dotenv()

os.environ["Cohere_API_KEY"] = os.getenv("COHERE_API_KEY")



In [9]:
### then we need to trenasform this data into chunks small docs
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_spliter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

documnts= text_spliter.split_documents(data)

In [ ]:
documnts 

[Document(metadata={'source': 'cricket_hist.txt'}, page_content="Cricket's origins can be traced back to the late 16th century in southeast England, with the earliest definite reference to the sport being in 1597. It evolved from earlier bat-and-ball games, potentially involving shepherds using their staffs to deflect a ball. The sport gained popularity in the 18th century, spreading throughout England and then to other parts of the British Empire. The first Laws of Cricket were codified in 1744, and the Marylebone Cricket Club (MCC) became the custodian of the Laws in 1787. International cricket emerged in the mid-19th century, with the first international match played in 1844 between the USA and Canada. \nHere's a more detailed look at the history:\nEarly Development (16th-18th Centuries):\nSaxon/Norman Roots:\nExperts believe cricket originated in the Weald region of southeast England during the Saxon or Norman periods. \nFirst References:\nThe earliest definite reference to the spo

In [25]:
### then we do embedding of this data and store it in vector database
from langchain.embeddings.cohere import CohereEmbeddings
from langchain.vectorstores import Chroma
from langchain.schema import Document


# Initialize Cohere embeddings
embedding = CohereEmbeddings(model="embed-english-v3.0", user_agent="langchain")
db = Chroma.from_documents(documnts, embedding)

In [26]:
db

In [27]:
quary = "what is cricket?"

result = db.similarity_search(quary)
print(result[0].page_content)

First References:
The earliest definite reference to the sport is from 1597, when it was mentioned in a legal deposition in Surrey.
Evolution from Other Games:
Cricket is thought to have evolved from earlier bat-and-ball games, possibly involving shepherds using their staffs to deflect a ball. 
Early Growth:
By the mid-17th century, village cricket became popular, and by the early 18th century, the game had spread to London and southeastern England. 
Codification of Laws:
The first formal set of Laws of Cricket were drawn up in 1744 and the MCC became the custodian in 1787. 
19th Century and Beyond:
International Cricket:
The first international match was played in 1844 between the USA and Canada. 
Global Expansion:
Cricket continued to spread, particularly with the British Empire, reaching countries like India, Australia, and the West Indies. 
Formal Organizations:
The Imperial Cricket Conference (later the ICC) was formed in 1909 to govern international cricket. 
Cricket in India:


In [51]:
### then we apply the LLM model to the Rag  to get the answer
from langchain_community.llms import Ollama

llm = Ollama(model="llama2")


In [52]:
##### then we define chatpromt template
from langchain_core.prompts import ChatPromptTemplate

promt =ChatPromptTemplate.from_template(
    "Answer the question based on the context below. If you don't know the answer, just say that you don't know, don't try to make up an answer. Context: {context} Question: {input}"

)

In [53]:
#### then we create the chain 
from langchain.chains.combine_documents import create_stuff_documents_chain

chain = create_stuff_documents_chain(llm ,promt)
chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template="Answer the question based on the context below. If you don't know the answer, just say that you don't know, don't try to make up an answer. Context: {context} Question: {input}"), additional_kwargs={})])
| Ollama()
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [54]:
## then we defin the retrieval  for retrive the data from the vector store

retriever = db.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'CohereEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000014DADCDF510>, search_kwargs={})

In [55]:
### then we need to create the retriever chain to get the input and passed to the llm
from langchain.chains import create_retrieval_chain
retriver_chain= create_retrieval_chain(retriever, chain)
retriver_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'CohereEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000014DADCDF510>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template="Answer the question based on the context below. If you don't know the answer, just say that you don't know, don't try to make up

In [58]:
### then we give the promt
result = retriver_chain.invoke({"input": "when cricket was invented?"})
print(result['answer'])


Based on the context provided, it is difficult to pinpoint an exact date or time period when cricket was invented. The earliest definite reference to the sport is from 1597, but it is believed that cricket evolved from earlier bat-and-ball games played in the Weald region of southeast England during the Saxon or Norman periods. Therefore, the exact origin of cricket can be traced back to the late 16th century, with the game potentially developing over time through a combination of cultural and historical factors.
